# Nhắc tư thế ngồi học đúng

**Sổ Nghiên Cứu AI, dự án mẫu 8/10.** Ước lượng góc cổ bằng MediaPipe Pose, đối chiếu với nhãn của người quan sát, và phân tích thí nghiệm có nhóm đối chứng.

**Câu hỏi nghiên cứu:** Nhắc nhở theo thời gian thực trong hai tuần có làm giảm thời gian ngồi sai tư thế của học sinh không?

**Giả thuyết:** Nhóm dùng ứng dụng có tỉ lệ thời gian ngồi sai giảm nhiều hơn nhóm đối chứng.

### Cách dùng notebook này
1. Trên Colab, chọn **Runtime → Change runtime type → T4 GPU** nếu có (không bắt buộc).
2. Chạy lần lượt từng ô bằng **Shift + Enter**. Lần đầu có thể mất vài phút để cài thư viện và tải dữ liệu.
3. Các ô **Việc của em** là chỗ em tự làm tiếp để biến dự án mẫu thành nghiên cứu của riêng mình.

**Nghiên cứu trên người:** cần giấy đồng ý của học sinh và phụ huynh (mẫu có trên trang Sổ Nghiên Cứu AI). Chỉ lưu góc đo, không lưu hình ảnh.

> **Liêm chính khoa học.** Đây là mã mẫu để luyện tập. Nếu dùng cho dự án dự thi, hãy ghi rõ trong sổ nghiên cứu và báo cáo đoạn mã nào lấy từ notebook này, đoạn nào em tự viết hoặc sửa. Dữ liệu dự thi phải do em thu thập hoặc lấy từ nguồn công khai có ghi nguồn.

In [ ]:
%pip install -q mediapipe

In [ ]:
import os, random, time, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")
SEED = 42
random.seed(SEED); np.random.seed(SEED)
plt.rcParams["figure.dpi"] = 110
def tieu_de(s): print("\n" + "=" * 8, s, "=" * 8)
import cv2, math, urllib.request
from scipy import stats

## Phần 1. Đo góc cổ từ ảnh chụp nghiêng
Góc cổ là góc giữa đường nối vai với tai và phương thẳng đứng. Cúi càng thấp, góc càng lớn. Tải ảnh lên thư mục `anh/` hoặc chụp bằng webcam ở ô dưới.

In [ ]:
try:
    from google.colab.output import eval_js
    from base64 import b64decode
    from IPython.display import Javascript, display
    def chup_anh(ten="anh/webcam.jpg"):
        os.makedirs("anh", exist_ok=True)
        display(Javascript("""async function chup(){const d=document.createElement('div');const b=document.createElement('button');b.textContent='Chụp';d.appendChild(b);
        const v=document.createElement('video');v.style.display='block';const s=await navigator.mediaDevices.getUserMedia({video:true});document.body.appendChild(d);d.appendChild(v);v.srcObject=s;await v.play();
        await new Promise(r=>b.onclick=r);const c=document.createElement('canvas');c.width=v.videoWidth;c.height=v.videoHeight;c.getContext('2d').drawImage(v,0,0);s.getVideoTracks()[0].stop();d.remove();return c.toDataURL('image/jpeg',0.9);}"""))
        open(ten, "wb").write(b64decode(eval_js("chup()").split(",")[1])); return ten
    print("Gọi chup_anh() trong một ô mới để chụp bằng webcam.")
except ImportError:
    print("Không chạy trên Colab: hãy đặt ảnh vào thư mục anh/.")

In [ ]:
def goc_co(duong_dan, det):
    import mediapipe as mp
    anh = mp.Image.create_from_file(duong_dan); r = det.detect(anh)
    if not r.pose_landmarks: return None
    p = r.pose_landmarks[0]; tai, vai = (p[7], p[11]) if p[7].visibility > p[8].visibility else (p[8], p[12])
    dx, dy = (tai.x - vai.x) * anh.width, (tai.y - vai.y) * anh.height
    return math.degrees(math.atan2(abs(dx), -dy))
if os.path.isdir("anh"):
    import mediapipe as mp
    from mediapipe.tasks import python as mpt
    from mediapipe.tasks.python import vision
    urllib.request.urlretrieve("https://storage.googleapis.com/mediapipe-models/pose_landmarker/pose_landmarker_lite/float16/1/pose_landmarker_lite.task", "pose.task")
    det = vision.PoseLandmarker.create_from_options(vision.PoseLandmarkerOptions(base_options=mpt.BaseOptions(model_asset_path="pose.task")))
    for f in sorted(os.listdir("anh")):
        g = goc_co(os.path.join("anh", f), det); print(f, "→", "không thấy người" if g is None else f"góc cổ {g:.1f}°")
else:
    print("Chưa có thư mục anh/.")

## Phần 2. Ngưỡng nào thì gọi là ngồi sai?
Nhờ một người quan sát gán nhãn 'đúng' hoặc 'sai' cho từng khung hình **mà không xem góc đo**. File `nhan_tu_the.csv` gồm cột `goc_co, nhan_nguoi` (1 = sai). Nếu chưa có, ô dưới dùng dữ liệu giả lập.

In [ ]:
from sklearn.metrics import cohen_kappa_score
if os.path.exists("nhan_tu_the.csv"): N = pd.read_csv("nhan_tu_the.csv"); NGUON = "của em"
else:
    g = np.r_[np.random.normal(18, 7, 180), np.random.normal(38, 9, 120)]; N = pd.DataFrame({"goc_co": g, "nhan_nguoi": (g + np.random.normal(0, 6, 300) > 28).astype(int)}); NGUON = "GIẢ LẬP"
nguong = np.arange(15, 45)
kappa = [cohen_kappa_score(N.nhan_nguoi, (N.goc_co > t).astype(int)) for t in nguong]
tot = nguong[int(np.argmax(kappa))]
print(NGUON, f"| ngưỡng tốt nhất {tot}° | kappa = {max(kappa):.2f} | độ chính xác = {((N.goc_co > tot) == N.nhan_nguoi).mean():.2f}")
plt.plot(nguong, kappa); plt.xlabel("ngưỡng góc cổ (độ)"); plt.ylabel("Cohen's kappa với người quan sát"); plt.show()

## Phần 3. Phân tích thí nghiệm hai nhóm
File `thi_nghiem.csv` gồm `hoc_sinh, nhom (app/doi_chung), truoc, sau` với `truoc`, `sau` là % thời gian ngồi sai trong tuần trước và tuần cuối. Phân nhóm **ngẫu nhiên** trước khi bắt đầu.

In [ ]:
if os.path.exists("thi_nghiem.csv"): T = pd.read_csv("thi_nghiem.csv"); NGUON = "của em"
else:
    n = 20; T = pd.DataFrame({"hoc_sinh": range(2 * n), "nhom": ["app"] * n + ["doi_chung"] * n, "truoc": np.random.normal(42, 10, 2 * n)})
    T["sau"] = T.truoc - np.where(T.nhom == "app", 9, 2) + np.random.normal(0, 7, 2 * n); NGUON = "GIẢ LẬP"
T["giam"] = T.truoc - T.sau
a, b = T[T.nhom == "app"].giam, T[T.nhom == "doi_chung"].giam
t, p = stats.ttest_ind(a, b, equal_var=False); u, pu = stats.mannwhitneyu(a, b)
d = (a.mean() - b.mean()) / np.sqrt((a.var() + b.var()) / 2)
print(NGUON, f"| giảm TB: app {a.mean():.1f} điểm %, đối chứng {b.mean():.1f} điểm %")
print(f"Welch t = {t:.2f}, p = {p:.4f} | Mann-Whitney p = {pu:.4f} | cỡ hiệu ứng Cohen d = {d:.2f}")
plt.boxplot([a, b]); plt.xticks([1, 2], ["Dùng ứng dụng", "Đối chứng"]); plt.ylabel("Mức giảm % thời gian ngồi sai"); plt.show()

## Việc của em
1. Gán nhãn ít nhất 300 khung hình với 2 người quan sát độc lập, tính kappa giữa hai người trước.
2. Tuyển 20 đến 40 tình nguyện viên, chia ngẫu nhiên hai nhóm, đo một tuần trước và một tuần sau.
3. Thử nghiệm các kiểu nhắc: âm thanh, đổi màu màn hình, rung. Đây là điểm mới đáng làm.